# SMILES 2026 Method 2 Colab Runner

This notebook updates the GitHub repo in a Drive-backed workspace, installs dependencies, and runs the adapted ICR Probe on `dataset.csv` with `Qwen/Qwen2.5-0.5B`.

Default Method 2 settings:

- all heads kept
- mean pooling across heads
- fixed `top_k = 10`
- z-score normalization before softmax
- paper-style MLP: `L -> 128 -> 64 -> 32 -> 1`
- `BatchNorm1d`, `LeakyReLU(0.01)`, `Dropout(p=0.3)`, sigmoid output
- notebook runs both `l2` and `l1 + l2` regularization variants


In [ ]:
from google.colab import drive

drive.mount('/content/drive', force_remount=False)


In [ ]:
import os
import subprocess
from pathlib import Path

TARGET_FOLDER = Path('/content/drive/MyDrive/hallucination_detection')
REPO_URL = 'https://github.com/olgafilimonova2004/hallucination_detection_draft.git'
REPO_NAME = 'hallucination_detection_draft'
REPO_PATH = TARGET_FOLDER / REPO_NAME
AUTO_STASH = True

TARGET_FOLDER.mkdir(parents=True, exist_ok=True)

def run(cmd: str, cwd: Path | None = None, check: bool = True) -> subprocess.CompletedProcess:
    print(f'$ {cmd}')
    result = subprocess.run(
        cmd,
        shell=True,
        cwd=str(cwd) if cwd is not None else None,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    print(result.stdout)
    if check and result.returncode != 0:
        raise RuntimeError(f'command failed with exit code {result.returncode}: {cmd}')
    return result

print('TARGET_FOLDER =', TARGET_FOLDER)
print('REPO_PATH =', REPO_PATH)

if not REPO_PATH.exists():
    run(f'git clone {REPO_URL}', cwd=TARGET_FOLDER)

run('git remote -v', cwd=REPO_PATH)
run('git branch --show-current', cwd=REPO_PATH)
run('git fetch origin', cwd=REPO_PATH)
status = run('git status --short', cwd=REPO_PATH, check=False).stdout.strip()

if status:
    print('Local changes detected.')
    if AUTO_STASH:
        run('git stash push -u -m "colab-auto-stash"', cwd=REPO_PATH)
    else:
        raise RuntimeError('Repo is dirty. Set AUTO_STASH = True or clean it manually.')

run('git pull --ff-only origin main', cwd=REPO_PATH)
run('git log --oneline -1', cwd=REPO_PATH)

os.chdir(REPO_PATH)
print('cwd =', Path.cwd())


In [ ]:
run('pip install -q -r requirements.txt', cwd=REPO_PATH)


In [ ]:
import torch

print('torch.cuda.is_available() =', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU =', torch.cuda.get_device_name(0))
else:
    print('GPU not available. Switch Colab runtime to GPU.')


## Method 2 runs

Method 2 needs eager attention because Qwen does not expose per-layer attentions under the default SDPA path. The runner handles that internally.

Use `batch-size = 1` first. Once the runtime is stable on your Colab GPU, you can try increasing it.

The first full run below uses the paper-style MLP with `l2` regularization only. The next run adds `l1` on top of `l2`.


In [ ]:
run(
    'python method2_icr_probe/run_method2.py '
    '--subset 40 '
    '--classifier mlp '
    '--hidden-dims 128,64,32 '
    '--dropout-p 0.3 '
    '--l2-weight-decay 1e-4 '
    '--batch-size 1 '
    '--max-length 256 '
    '--cache-dtype float32',
    cwd=REPO_PATH,
)


In [ ]:
run(
    'python method2_icr_probe/run_method2.py '
    '--classifier mlp '
    '--hidden-dims 128,64,32 '
    '--dropout-p 0.3 '
    '--l2-weight-decay 1e-4 '
    '--batch-size 1 '
    '--cache-dtype float32',
    cwd=REPO_PATH,
)


In [ ]:
run(
    'python method2_icr_probe/run_method2.py '
    '--classifier mlp '
    '--hidden-dims 128,64,32 '
    '--dropout-p 0.3 '
    '--l1-lambda 1e-5 '
    '--l2-weight-decay 1e-4 '
    '--batch-size 1 '
    '--cache-dtype float32 '
    '--output-file method2_icr_probe/artifacts/method2_l1_l2.json',
    cwd=REPO_PATH,
)


In [ ]:
from pathlib import Path

artifacts_dir = REPO_PATH / 'method2_icr_probe' / 'artifacts'
for name in [
    'method2_results.json',
    'method2_results_metadata.json',
    'method2_l1_l2.json',
    'method2_l1_l2_metadata.json',
]:
    path = artifacts_dir / name
    if path.exists():
        print(f'===== {name} =====')
        print(path.read_text()[:6000])
        print()
